# MPP Aluminium — AE / EMDE / China intensity pathway derivation

Derives two per-tonne-Al intensity pathways from MPP's 1.5DS asset-level outputs, 2020–2050:

1. **Process intensity** (tCO₂e/tAl): direct emissions excluding electricity — smelter process CO₂ + PFCs + anode thermal + refinery digestion/calcination fuel.
2. **Total intensity** (tCO₂e/tAl): process + electricity (Scope 2 imported + captive fossil combustion).

Bucketing: every smelter and refinery asset is joined to its country (via composite key `region + technology + capacity`), then countries map to AE / EMDE / China per `ALU_Region_mapping.csv`.

Refinery emissions are attributed to smelter regions **bucket-locally** (AE refineries → AE smelters, etc.) — see prior discussion for tradeoffs vs global adder.

Direct-emissions coefficients (Tech Appendix Exhibit TA3.3, page 11):

| Anode archetype | tCO₂e/tAl |
|---|---|
| Carbon Anode (Hall-Héroult) | 2.13 |
| Carbon Anode + CCS | 1.17 |
| Inert Anode | 0.10 |

Note: MPP's `co2_scope1` column is actually **CO₂e** — it bundles anode process CO₂, PFCs, and anode thermal (verified against Exhibit TA3.3).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(".")
MPP = DATA_DIR / "mpp-shared-code" / "aluminium" / "data" / "lc"

# archetype constants (tCO2e/tAl) from Tech Appendix Exhibit TA3.3
COEFF = {"CA": 2.13, "CA_CCS": 1.17, "IA": 0.10}

# alumina-to-aluminium ratio (kg alumina per t Al) → dimensionless multiplier
ALUMINA_PER_AL = 1.935

In [ ]:
# --- region mapping: country → AE / EMDE / China ---
raw = pd.read_csv(DATA_DIR / "ALU_Region_mapping.csv")
country_to_bucket = {}
for c in raw["Advanced_Economies"].dropna(): country_to_bucket[c] = "AE"
for c in raw["Emerging_Economies"].dropna(): country_to_bucket[c] = "EMDE"
for c in raw["China"].dropna(): country_to_bucket[c] = "China"

# aliases (MPP asset stack sometimes uses different spellings than the IEA list)
ALIASES = {
    "South Korea": "Korea",
    "USA": "United States",
    "Czechia": "Czech Republic",
    "Slovakia": "Slovak Republic",
}
# explicit overrides for countries missing from the mapping csv
EXTRA = {"Iran": "EMDE"}

def bucket_of(country, region=None):
    """Return AE / EMDE / China / None for a country string.
    Handles two MPP data quirks: (1) Chinese smelter rows have country set to the
    sub-region name (e.g. 'China - North West'); (2) Chinese refinery rows have
    country='Refinery' as a placeholder, so we fall back to region."""
    if country and str(country).startswith("China"):
        return "China"
    if country == "Refinery" and region and str(region).startswith("China"):
        return "China"
    c = ALIASES.get(country, country)
    if c in country_to_bucket: return country_to_bucket[c]
    if c in EXTRA:             return EXTRA[c]
    if region and str(region).startswith("China"): return "China"
    return None

print(f"AE: {sum(v=='AE' for v in country_to_bucket.values())} countries")
print(f"EMDE: {sum(v=='EMDE' for v in country_to_bucket.values())} countries")
print(f"China: {sum(v=='China' for v in country_to_bucket.values())} countries")

In [ ]:
# --- SMELTERS: join country onto every asset-year row ---
# Composite-key trick: initial_asset_stack has country, plant_stack has uuid.
# Both share (region, technology, annual_production_capacity). We use a
# within-group counter to disambiguate assets with identical keys.

s_init = pd.read_csv(MPP / "def" / "intermediate" / "initial_asset_stack.csv")
s_plant = pd.read_csv(MPP / "def" / "final" / "plant_stack_transition_aluminium_lc_def.csv")

KEY = ["region", "technology", "annual_production_capacity"]

# match 2020 plant rows to initial rows to get uuid ↔ country
s_init = s_init.copy(); s_plant2020 = s_plant[s_plant.year == 2020].copy()
s_init["_k"] = s_init.groupby(KEY).cumcount()
s_plant2020["_k"] = s_plant2020.groupby(KEY).cumcount()
uuid_country = (s_plant2020.merge(s_init[KEY + ["country", "_k"]], on=KEY + ["_k"], how="left")
                            [["uuid", "country"]])
assert uuid_country.country.notna().all(), "some 2020 assets did not match initial stack"

# for greenfield uuids added after 2020: assign country by region majority
uuid_country = uuid_country.set_index("uuid")
s_plant["country"] = s_plant["uuid"].map(uuid_country["country"])
missing_uuids = s_plant[s_plant.country.isna()].uuid.unique()
print(f"initial-stack uuids: {len(uuid_country)}, greenfield uuids needing region fallback: {len(missing_uuids)}")

# for greenfields, pick the modal country of each region among tagged assets
region_modal_country = (s_plant.dropna(subset=["country"]).groupby("region")["country"]
                        .agg(lambda s: s.value_counts().index[0]))
s_plant["country"] = s_plant["country"].fillna(s_plant["region"].map(region_modal_country))
s_plant["bucket"] = s_plant.apply(lambda r: bucket_of(r["country"], r["region"]), axis=1)
assert s_plant.bucket.notna().all(), f"unmapped countries: {s_plant[s_plant.bucket.isna()].country.unique()}"
print("smelter country → bucket coverage: OK")

In [ ]:
# --- anode archetype classification ---
def archetype(tech):
    if tech.startswith("Carbon Anode+CCS"): return "CA_CCS"
    if tech.startswith("Carbon Anode"):     return "CA"
    if tech.startswith("Inert Anode"):      return "IA"
    raise ValueError(tech)

s_plant["archetype"] = s_plant["technology"].map(archetype)
s_plant["coeff"]     = s_plant["archetype"].map(COEFF)
# process emissions per asset-year (Mt CO2e = Mt Al × tCO2/tAl)
s_plant["process_Mt"] = s_plant["annual_production_volume"] * s_plant["coeff"]

# aggregate to bucket-year
smelter = (s_plant.groupby(["bucket", "year"])
                  .agg(process_Mt=("process_Mt", "sum"),
                       production_Mt=("annual_production_volume", "sum"))
                  .reset_index())
smelter["smelter_process_intensity"] = smelter.process_Mt / smelter.production_Mt
smelter.head()

In [ ]:
# --- REFINERIES: same join pattern, plus emissions from interface_outputs ---
r_init = pd.read_csv(MPP / "def_refineries" / "intermediate" / "initial_asset_stack.csv")
r_plant = pd.read_csv(MPP / "def_refineries" / "final" / "plant_stack_transition_aluminium_lc_def_refineries.csv")

r_init = r_init.copy(); r_plant2020 = r_plant[r_plant.year == 2020].copy()
r_init["_k"] = r_init.groupby(KEY).cumcount()
r_plant2020["_k"] = r_plant2020.groupby(KEY).cumcount()
r_uuid_country = (r_plant2020.merge(r_init[KEY + ["country", "_k"]], on=KEY + ["_k"], how="left")
                              [["uuid", "country"]].set_index("uuid"))
r_plant["country"] = r_plant["uuid"].map(r_uuid_country["country"])
r_region_modal = (r_plant.dropna(subset=["country"]).groupby("region")["country"]
                        .agg(lambda s: s.value_counts().index[0]))
r_plant["country"] = r_plant["country"].fillna(r_plant["region"].map(r_region_modal))
r_plant["bucket"] = r_plant.apply(lambda r: bucket_of(r["country"], r["region"]), axis=1)
assert r_plant.bucket.notna().all(), f"unmapped countries: {r_plant[r_plant.bucket.isna()].country.unique()}"
print("refinery country → bucket coverage: OK")

In [ ]:
# --- refinery emissions per asset via region-tech-year intensity ---
r_iface = pd.read_csv(MPP / "def_refineries" / "final" / "interface_outputs_aluminium_lc_def_refineries.csv")

def wide_intensity(parameter):
    # returns a df keyed on (region, technology, year) with value / production intensity
    val = r_iface[r_iface.parameter == parameter][["region","technology","year","value"]].rename(columns={"value":"num"})
    prod = r_iface[r_iface.parameter == "Annual production volume"][["region","technology","year","value"]].rename(columns={"value":"denom"})
    m = val.merge(prod, on=["region","technology","year"])
    m["intensity"] = m["num"] / m["denom"].where(m["denom"] > 0)
    return m[["region","technology","year","intensity"]]

r_scope1_int = wide_intensity("CO2 Scope1").rename(columns={"intensity":"scope1_int"})
r_plant = r_plant.merge(r_scope1_int, on=["region","technology","year"], how="left")
r_plant["scope1_Mt"] = r_plant["annual_production_volume"] * r_plant["scope1_int"].fillna(0)

# aggregate to bucket-year
refinery = (r_plant.groupby(["bucket","year"])
                   .agg(refinery_process_Mt=("scope1_Mt", "sum"),
                        alumina_Mt=("annual_production_volume", "sum"))
                   .reset_index())
refinery["refinery_intensity_per_tAa"] = refinery.refinery_process_Mt / refinery.alumina_Mt
refinery.head()

In [ ]:
# --- combine into process intensity per tonne Al ---
# Bucket-local attribution: each bucket's smelters draw alumina from that bucket's refineries.
# Refinery contribution to per-tAl intensity = refinery_intensity_per_tAa × 1.935 kg alumina / kg Al

pathway = smelter.merge(refinery[["bucket","year","refinery_intensity_per_tAa"]], on=["bucket","year"], how="left")
pathway["refinery_intensity_per_tAl"] = pathway["refinery_intensity_per_tAa"] * ALUMINA_PER_AL
pathway["process_intensity"] = pathway["smelter_process_intensity"] + pathway["refinery_intensity_per_tAl"]
pathway[["bucket","year","smelter_process_intensity","refinery_intensity_per_tAl","process_intensity"]].head()

In [ ]:
# --- electricity intensity (for notebook 2) ---
# For each smelter asset, split total emissions into process vs electricity buckets.
# Scope 2 in interface_outputs already excludes captive. Captive = Scope 1 - archetype coefficient.

s_iface = pd.read_csv(MPP / "def" / "final" / "interface_outputs_aluminium_lc_def.csv")
s_scope1 = wide_intensity_s = None  # use local helper against s_iface

def sm_int(parameter):
    val = s_iface[s_iface.parameter == parameter][["region","technology","year","value"]].rename(columns={"value":"num"})
    prod = s_iface[s_iface.parameter == "Annual production volume"][["region","technology","year","value"]].rename(columns={"value":"denom"})
    m = val.merge(prod, on=["region","technology","year"])
    m["intensity"] = m["num"] / m["denom"].where(m["denom"] > 0)
    return m[["region","technology","year","intensity"]]

s_scope1_int = sm_int("CO2 Scope1").rename(columns={"intensity":"s1_int"})
s_scope2_int = sm_int("CO2 Scope2").rename(columns={"intensity":"s2_int"})
s_elec_int   = sm_int("Electricity Consumption").rename(columns={"intensity":"elec_int"})  # MWh/tAl (mislabelled GJ)

s2 = s_plant.merge(s_scope1_int, on=["region","technology","year"], how="left") \
            .merge(s_scope2_int, on=["region","technology","year"], how="left") \
            .merge(s_elec_int,   on=["region","technology","year"], how="left")

s2["captive_int"]        = (s2["s1_int"] - s2["coeff"]).clip(lower=0)   # tCO2/tAl from captive fossil fuel
s2["elec_emissions_int"] = s2["scope2_intensity"] if False else (s2["s2_int"] + s2["captive_int"])  # tCO2/tAl
s2["elec_emissions_Mt"]  = s2["annual_production_volume"] * s2["elec_emissions_int"]
s2["elec_TWh"]           = s2["annual_production_volume"] * s2["elec_int"]  # Mt Al × MWh/tAl = million MWh = TWh

elec = (s2.groupby(["bucket","year"])
          .agg(elec_emissions_Mt=("elec_emissions_Mt","sum"),
               elec_TWh=("elec_TWh","sum"))
          .reset_index())
# gCO2/kWh = tCO2/MWh × 1000 = (Mt / TWh) × 1000
elec["power_intensity_gCO2_per_kWh"] = elec["elec_emissions_Mt"] / elec["elec_TWh"] * 1000
elec.head()

In [ ]:
# --- total intensity (process + electricity per tAl) and export ---
# For total: convert electricity emissions to per-tAl using smelter production
pathway = pathway.merge(elec[["bucket","year","elec_emissions_Mt","elec_TWh","power_intensity_gCO2_per_kWh"]], on=["bucket","year"], how="left")
pathway["elec_intensity_per_tAl"] = pathway["elec_emissions_Mt"] / pathway["production_Mt"]
pathway["total_intensity"] = pathway["process_intensity"] + pathway["elec_intensity_per_tAl"]

out_cols = ["bucket","year","production_Mt","smelter_process_intensity",
            "refinery_intensity_per_tAl","process_intensity",
            "elec_intensity_per_tAl","total_intensity",
            "elec_TWh","power_intensity_gCO2_per_kWh"]
pathway[out_cols].to_csv(DATA_DIR / "mpp_al_intensity_pathway_1p5DS.csv", index=False)
pathway[out_cols].round(3).head(20)